# LayerCake — Interactive Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yoder23/layercake/blob/master/notebooks/layercake_demo.ipynb)
[![GitHub](https://img.shields.io/badge/github-Yoder23%2Flayercake-black)](https://github.com/Yoder23/layercake)

**What this notebook shows:**

1. **Bit-exact paste proof** — domain module copied between models, `max_diff = 0.0`
2. **Generation identity** — 50-token sequence is token-for-token identical after paste
3. **Cross-size portability** — paste a domain module from a 48M model into a 150M model
4. **LoRA comparison** — why LoRA can't do this
5. **Mini domain training** — train a tiny domain module for 200 steps, then paste it

> **No GPU required.** The paste proofs and domain training demo use tiny synthetic models and run in ~30 seconds on CPU.

---

## Setup

In [ ]:
# Install LayerCake (takes ~30 seconds)
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!git clone -q https://github.com/Yoder23/layercake
import sys, os
sys.path.insert(0, 'layercake')
os.chdir('layercake')
print('✓ Ready')

---
## Part 1 — The Paste Proof

Run `verify_paste.py` — the same script that runs in CI on every commit.
This requires no data and no pretrained checkpoints.

In [ ]:
!python verify_paste.py

---
## Part 2 — Why It Works: The ABI Bottleneck

```
48M model (d_model=512)              150M model (d_model=768)
─────────────────────────────────────────────────────────────────
core_blocks      [512×512]           core_blocks      [768×768]
core_to_abi   (512 → d_abi=512)      core_to_abi   (768 → d_abi=512)
                    ↓                                    ↓
        domain_module [512×512]  ←── SAME TENSOR ──→  domain_module [512×512]
                    ↓                                    ↓
abi_to_core   (512 → 512)            abi_to_core   (512 → 768)
```

The domain module only ever sees `d_abi=512` vectors — it is **independent of `d_model`**.
Paste is a direct memory copy: `W_pasted = W_source` (exact clone).

**Why LoRA can't do this:**  
LoRA decomposes weight updates as `ΔW = AB` where `A` is `(d_model, rank)`.  
The shape depends on `d_model`, so a LoRA trained on a 512-wide model has tensors
that don't fit a 768-wide model. There's no well-defined paste operation.

| | LoRA | LayerCake domain module |
|---|---|---|
| Portable across model sizes? | ❌ No | ✅ Yes |
| Paste is bit-exact? | ❌ No | ✅ Yes |
| Generation-identical after paste? | ❌ No | ✅ Yes |

---
## Part 3 — Live Demo: Train a Domain Module and Paste It

We'll:
1. Build a small 48M LayerCake model
2. Generate synthetic "chess" domain data
3. Train a domain module for 200 steps (~10 seconds)
4. Paste it into a fresh 150M model
5. Prove the pasted module produces identical outputs

In [ ]:
import torch
import torch.nn.functional as F
from model import LayerCakeLMFixedABI

VOCAB = 1000; D_ABI = 512; SEQ = 64; BATCH = 8
DOMAINS = ['chess', 'python']

def build(d_model, n_layers, n_heads):
    return LayerCakeLMFixedABI(
        vocab_size=VOCAB, d_model=d_model, d_abi=D_ABI,
        n_core_layers=n_layers, n_heads=n_heads, d_ff=d_model*4,
        domain_names=DOMAINS, max_seq_len=SEQ,
        use_router=False, domain_module_type='lite')

torch.manual_seed(42)
model_48M = build(512, 4, 8)   # Our 'trained' small model
model_150M = build(768, 6, 12)  # Target large model

total_48M = sum(p.numel() for p in model_48M.parameters())
total_150M = sum(p.numel() for p in model_150M.parameters())
domain_params = sum(p.numel() for n, p in model_48M.named_parameters() if 'domain_modules.chess' in n)
print(f'48M model:    {total_48M/1e6:.1f}M params')
print(f'150M model:   {total_150M/1e6:.1f}M params')
print(f'Chess domain: {domain_params/1e6:.2f}M params (same in both)')

In [ ]:
# Freeze the core, train only the chess domain module
for name, p in model_48M.named_parameters():
    p.requires_grad = 'domain_modules.chess' in name

opt = torch.optim.Adam(
    [p for p in model_48M.parameters() if p.requires_grad], lr=1e-3)

mask = torch.zeros(BATCH, len(DOMAINS))
mask[:, DOMAINS.index('chess')] = 1.0

print('Training chess domain module (200 steps)...')
for step in range(200):
    x = torch.randint(0, VOCAB, (BATCH, SEQ))
    logits, _ = model_48M(x, domain_mask=mask)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), x[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f'  step {step:3d}  loss={loss.item():.3f}')

print(f'\nFinal loss: {loss.item():.3f}')
print('Core frozen throughout — only chess domain module updated ✓')

In [ ]:
# Paste the trained chess domain from 48M → 150M
src = model_48M.state_dict()
tgt = model_150M.state_dict()
prefix = 'domain_modules.chess.'
copied = 0
for k, v in src.items():
    if k.startswith(prefix):
        tgt[k] = v.clone()
        copied += 1
model_150M.load_state_dict(tgt)

print(f'Pasted {copied} tensors from 48M → 150M')

# Verify: domain module function is identical on same d_abi inputs
h = torch.randn(BATCH, SEQ, D_ABI)
model_48M.eval(); model_150M.eval()
with torch.no_grad():
    out_48M  = model_48M.domain_modules['chess'](h)
    out_150M = model_150M.domain_modules['chess'](h)

max_diff = (out_48M - out_150M).abs().max().item()
print(f'\nDomain module output max_diff: {max_diff:.2e}')
print('✓ PASS — bit-exact paste' if max_diff == 0.0 else '✗ FAIL')

---
## Next Steps

- **Train your own core:** [`train_core.py`](https://github.com/Yoder23/layercake/blob/master/train_core.py)
- **Train a real domain module:** [`train_domain.py`](https://github.com/Yoder23/layercake/blob/master/train_domain.py)
- **Read the claims:** [`CLAIMS.md`](https://github.com/Yoder23/layercake/blob/master/CLAIMS.md)
- **Read the scope:** [`SKEPTICS.md`](https://github.com/Yoder23/layercake/blob/master/SKEPTICS.md)
- **Cross-model alignment (different seeds/architectures):** [ABI repo](https://github.com/Yoder23/abi)